# Punto 2 – CNN sobre Fashion-MNIST: desde cero, VGG16 y MobileNetV2

**Taller 1 – Aprendizaje Profundo** · Maestría en Inteligencia Artificial, Pontificia Universidad Javeriana

Desarrollo del Punto 2 por **Andrea Barraza**. Las tres aproximaciones viven en un solo notebook a
propósito: comparten partición, semilla, lotes, callbacks y métricas en una misma corrida, que es lo
que pide la rúbrica cuando habla de *comparación justa bajo condiciones documentadas*.

Orden del notebook:

1. **Datos y diseño experimental** (sección 1): carga, revisión de integridad, partición estratificada
   54 000 / 6 000 / 10 000 y pipeline `tf.data` común a las tres familias.
2. **Arquitecturas** (sección 2): CNN propia, VGG16 y MobileNetV2, las tres recibiendo la imagen
   original de 28×28×1. El redimensionamiento a 96×96, la conversión a RGB y el preprocesamiento de
   ImageNet ocurren *dentro* de cada modelo, así que no hay dos formatos de datos conviviendo.
3. **Búsqueda de hiperparámetros** (sección 3): presupuesto común, se varían capacidad, dropout,
   aumentación y learning rate.
4. **Entrenamiento final y fine-tuning** (sección 4) y **evaluación sobre test** (sección 5), que se
   toca una sola vez.
5. **Análisis de errores e incertidumbre** (sección 6) y guía de redacción (sección 7).

Requiere **GPU** (en Colab: *Entorno de ejecución > Cambiar tipo de entorno > T4*). Con
`MODO_RAPIDO = False` la corrida completa toma entre hora y media y dos horas y media; VGG16 se lleva
la mayor parte. Los resultados se guardan en `results/punto2/` a medida que avanzan, de modo que una
desconexión no obliga a empezar de cero.


## 0. Preparación del entorno

In [ ]:
import gc
import json
import os
import random
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from IPython.display import display
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             log_loss, precision_recall_fscore_support)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

SEED = 42
MODO_RAPIDO = False          # True solo para depurar: subconjunto y pocas épocas.
IMG_SIZE = 96                # tamaño al que se lleva la imagen para las redes de ImageNet
NUM_CLASSES = 10

HAY_GPU = bool(tf.config.list_physical_devices("GPU"))
BATCH_SIZE = 128 if HAY_GPU else 64
EPOCHS_SEARCH = 4 if MODO_RAPIDO else 12
EPOCHS_FINAL = 8 if MODO_RAPIDO else 30
AUTOTUNE = tf.data.AUTOTUNE


def seed_all(s=SEED):
    """Se llama antes de construir cada modelo para que todas las corridas arranquen igual."""
    os.environ["PYTHONHASHSEED"] = str(s)
    random.seed(s)
    np.random.seed(s)
    tf.random.set_seed(s)


seed_all()
print("TensorFlow:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
print("MODO_RAPIDO:", MODO_RAPIDO, "| épocas búsqueda:", EPOCHS_SEARCH, "| épocas final:", EPOCHS_FINAL)
if not HAY_GPU:
    print("AVISO: sin GPU esto tarda horas. Entorno de ejecución > Cambiar tipo de entorno > T4.")


In [ ]:
# En Colab clonamos el repo para dejar los resultados en results/punto2.
REPO_URL = "https://github.com/DANIEL-VELASCO/taller1-deep-learning.git"
EN_COLAB = "google.colab" in sys.modules
print("Colab:", EN_COLAB)

if EN_COLAB:
    if not Path("/content/taller1-deep-learning").exists():
        !git clone -q {REPO_URL} /content/taller1-deep-learning
    RAIZ = Path("/content/taller1-deep-learning")
else:
    RAIZ = Path.cwd().resolve().parents[1]     # el notebook vive en code/punto2_cnn_fashion_mnist/

RUTA_RES = RAIZ / "results" / "punto2"
RUTA_RES.mkdir(parents=True, exist_ok=True)

# Los checkpoints y los modelos pesados se quedan FUERA del repo: un VGG16 guardado son unos
# 56 MB y GitHub avisa a partir de 50 MB. Al repo van las tablas, las figuras y la CNN pequeña.
RUTA_TMP = Path("/content/checkpoints") if EN_COLAB else Path("checkpoints_tmp")
RUTA_TMP.mkdir(parents=True, exist_ok=True)

print("resultados  ->", RUTA_RES)
print("temporales  ->", RUTA_TMP)


## 1. Datos y diseño experimental

Fashion-MNIST contiene 60 000 imágenes de desarrollo y 10 000 de test, de 28×28 píxeles en escala de grises y 10 clases. Se separa el desarrollo en 54 000 entrenamiento y 6 000 validación de forma estratificada. El test no participa en la búsqueda. Todas las familias usan la misma partición, semilla, criterio de parada y métricas. Las redes ImageNet requieren conversión a RGB y redimensionamiento.


In [ ]:
(x_dev, y_dev), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
classes = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
           "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

# Revisión de integridad antes de partir nada.
print("desarrollo:", x_dev.shape, x_dev.dtype, "| test:", x_test.shape, x_test.dtype)
print("rango de píxeles:", int(x_dev.min()), "-", int(x_dev.max()))
assert x_dev.dtype == np.uint8 and x_test.dtype == np.uint8   # uint8: no puede haber NaN
assert x_dev.min() == 0 and x_dev.max() == 255
assert sorted(np.unique(y_dev)) == list(range(NUM_CLASSES))
assert len(x_dev) == 60000 and len(x_test) == 10000

# Fashion-MNIST trae imágenes repetidas. Las contamos para poder mencionarlo en el informe,
# pero NO las quitamos: el benchmark estándar usa las 70 000 y eliminarlas rompería la
# comparabilidad con los resultados publicados.
vistas_dev = {v.tobytes() for v in x_dev}
repetidas_dev = len(x_dev) - len(vistas_dev)
fuga_test = sum(v.tobytes() in vistas_dev for v in x_test)
print("imágenes repetidas dentro de desarrollo:", repetidas_dev)
print("imágenes de test que también están en desarrollo:", fuga_test)

x_train, x_val, y_train, y_val = train_test_split(
    x_dev, y_dev, test_size=0.1, random_state=SEED, stratify=y_dev)

print(x_train.shape, x_val.shape, x_test.shape)
for nombre, y in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(nombre, np.bincount(y, minlength=NUM_CLASSES))
assert (len(x_train), len(x_val), len(x_test)) == (54000, 6000, 10000)

fig, ax = plt.subplots(2, 5, figsize=(12, 5))
for c, a in enumerate(ax.flat):
    i = np.where(y_train == c)[0][0]
    a.imshow(x_train[i], cmap="gray")
    a.set_title(classes[c])
    a.axis("off")
plt.tight_layout()
plt.savefig(RUTA_RES / "ejemplos_clases.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
def _a_float32(imagenes, etiquetas):
    return tf.cast(imagenes, tf.float32), etiquetas


def make_ds(x, y, training=False):
    """Pipeline único: entrega lotes de 28x28x1 en float32, iguales para las tres familias.

    Detalles que importan:
    - el canal se agrega con numpy (`x[..., None]` es una vista, no copia) en vez de con un
      `map` imagen por imagen;
    - el casteo a float32 se hace sobre el lote ya armado, no sobre cada imagen suelta;
    - `shuffle` va antes de `batch` para que los lotes cambien de composición cada época;
    - solo se cachean validación y test: cachear después de barajar congelaría el orden.
    """
    ds = tf.data.Dataset.from_tensor_slices((x[..., np.newaxis], y))
    if training:
        ds = ds.shuffle(len(x), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE).map(_a_float32, num_parallel_calls=AUTOTUNE)
    if not training:
        ds = ds.cache()
    return ds.prefetch(AUTOTUNE)


# Un solo juego de datasets para las tres familias. La adaptación a RGB, el redimensionamiento
# y el preprocesamiento de ImageNet viven dentro de cada modelo, así que aquí no hay nada
# específico de VGG16 ni de MobileNetV2 y las tres ven exactamente los mismos lotes.
DS = {
    "train": make_ds(x_train, y_train, training=True),
    "val": make_ds(x_val, y_val),
    "test": make_ds(x_test, y_test),
}

for particion, ds in DS.items():
    imagenes, etiquetas = next(iter(ds))
    print(particion, "->", imagenes.shape, imagenes.dtype, "| etiquetas", etiquetas.shape)
    assert imagenes.shape[1:] == (28, 28, 1) and imagenes.dtype == tf.float32


## 2. Arquitecturas

In [ ]:
def make_augmentation(name="data_augmentation"):
    """Devuelve una instancia NUEVA de aumentación para cada modelo.

    No se reutiliza una instancia global, porque Keras puede construirla con la
    forma del primer modelo y luego rechazar una forma diferente.
    """
    return tf.keras.Sequential(
        [
            tf.keras.layers.RandomTranslation(
                height_factor=0.08,
                width_factor=0.08,
                fill_mode="nearest",
                seed=SEED,
            ),
            tf.keras.layers.RandomRotation(
                factor=0.06,
                fill_mode="nearest",
                seed=SEED,
            ),
        ],
        name=name,
    )


def preprocesar(family, x):
    """Equivalente exacto de `preprocess_input`, escrito con capas que sí se serializan.

    La versión original usaba `Lambda(preprocess_input)`. Funciona, pero un modelo guardado
    con una capa Lambda solo se puede volver a cargar con `safe_mode=False`, y eso muerde
    cuando el informe se arma en otra sesión. Aquí se hace la misma cuenta con `Rescaling`.

    `x` entra con un solo canal (el gris, en 0-255) y sale con tres canales listos para el
    backbone. La celda siguiente verifica numéricamente que coincide con la función oficial.
    """
    if family == "vgg16":
        # Modo 'caffe': se pasa de RGB a BGR y se resta la media de ImageNet por canal. Como
        # los tres canales son copias del mismo gris, el intercambio RGB->BGR no cambia nada
        # y basta con restarle a cada copia la media que le corresponde.
        return tf.keras.layers.Concatenate(axis=-1, name="gris_a_bgr_centrado")([
            tf.keras.layers.Rescaling(1.0, offset=-103.939, name="canal_b")(x),
            tf.keras.layers.Rescaling(1.0, offset=-116.779, name="canal_g")(x),
            tf.keras.layers.Rescaling(1.0, offset=-123.680, name="canal_r")(x),
        ])
    # MobileNetV2 espera el rango [-1, 1].
    x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1.0, name="a_rango_m1_1")(x)
    return tf.keras.layers.Concatenate(axis=-1, name="gris_a_rgb")([x, x, x])


def cnn(cfg):
    """CNN desde cero. Recibe imágenes originales 28x28x1."""
    inputs = tf.keras.Input(shape=(28, 28, 1), name="fashion_mnist_input")
    x = inputs

    if cfg["aug"]:
        x = make_augmentation("augmentation_cnn")(x)

    # La normalización pertenece al modelo para mantener un pipeline uniforme.
    x = tf.keras.layers.Rescaling(1.0 / 255.0, name="normalization")(x)

    for block, n_filters in enumerate(cfg["filters"], start=1):
        x = tf.keras.layers.Conv2D(
            n_filters,
            kernel_size=3,
            padding="same",
            use_bias=False,
            kernel_initializer="he_normal",
            name=f"block_{block}_conv",
        )(x)
        x = tf.keras.layers.BatchNormalization(name=f"block_{block}_bn")(x)
        x = tf.keras.layers.Activation("relu", name=f"block_{block}_relu")(x)
        x = tf.keras.layers.MaxPool2D(name=f"block_{block}_pool")(x)

    x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
    x = tf.keras.layers.Dense(128, activation="relu", name="dense_features")(x)
    x = tf.keras.layers.Dropout(cfg["drop"], name="dropout")(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="classifier")(x)

    return tf.keras.Model(inputs, outputs, name="CNN_desde_cero")


def transfer(cfg, family):
    """VGG16 o MobileNetV2 adaptada internamente desde 28x28x1.

    Orden: entrada gris -> aumento -> redimensionamiento -> RGB + preprocesamiento
    -> backbone preentrenado -> clasificador de Fashion-MNIST.
    """
    inputs = tf.keras.Input(shape=(28, 28, 1), name="fashion_mnist_input")
    x = inputs

    # La aumentación ocurre antes de redimensionar y antes de convertir a RGB.
    if cfg["aug"]:
        x = make_augmentation(f"augmentation_{family}")(x)

    x = tf.keras.layers.Resizing(IMG_SIZE, IMG_SIZE, interpolation="bilinear", name="resize")(x)
    x = preprocesar(family, x)

    if family == "vgg16":
        base = tf.keras.applications.VGG16(
            include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
    elif family == "mobilenetv2":
        base = tf.keras.applications.MobileNetV2(
            include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
    else:
        raise ValueError(f"Familia no reconocida: {family}")

    base.trainable = False
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
    x = tf.keras.layers.Dropout(cfg["drop"], name="dropout")(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="classifier")(x)

    return tf.keras.Model(inputs, outputs, name=family)


def build(family, cfg):
    return cnn(cfg) if family == "cnn" else transfer(cfg, family)


def backbone_de(model):
    """El backbone es el único submodelo dentro del modelo.

    Se busca por tipo en vez de guardarlo en un atributo (`model.backbone`), porque un
    atributo así no sobrevive a guardar y recargar el modelo.
    """
    return next(capa for capa in model.layers if isinstance(capa, tf.keras.Model)
                and not isinstance(capa, tf.keras.Sequential))


def compile_m(model, learning_rate):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )


def cbs(tag=None):
    """Mismos callbacks para las tres familias.

    `EarlyStopping` ya devuelve los mejores pesos, así que durante la búsqueda no se guarda
    checkpoint: eran hasta 56 MB por corrida de VGG16 y solo hacían más lento el entrenamiento.
    Las corridas finales sí dejan checkpoint, pero fuera del repositorio.
    """
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7),
    ]
    if tag is not None:
        callbacks.append(tf.keras.callbacks.ModelCheckpoint(
            str(RUTA_TMP / f"{tag}.keras"), monitor="val_loss", save_best_only=True))
    return callbacks


In [ ]:
# Dos comprobaciones antes de gastar horas de GPU.

# 1) El preprocesamiento propio debe coincidir con las funciones oficiales de Keras.
gris = x_test[:8, :, :, np.newaxis].astype("float32")
gris = tf.image.resize(gris, (IMG_SIZE, IMG_SIZE), method="bilinear").numpy()

for family, oficial in [("vgg16", tf.keras.applications.vgg16.preprocess_input),
                        ("mobilenetv2", tf.keras.applications.mobilenet_v2.preprocess_input)]:
    tf.keras.backend.clear_session()
    entrada = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 1))
    sonda = tf.keras.Model(entrada, preprocesar(family, entrada))
    obtenido = sonda.predict(gris, verbose=0)
    esperado = oficial(np.repeat(gris, 3, axis=-1).copy())
    print(f"{family}: diferencia máxima contra preprocess_input = {np.abs(obtenido - esperado).max():.2e}")
    assert np.allclose(obtenido, esperado, atol=1e-4)

# 2) Las tres familias deben aceptar 28x28x1 y devolver 10 probabilidades.
shape_test_configs = {
    "cnn": {"filters": (32, 64), "drop": 0.3, "aug": True, "lr": 1e-3},
    "vgg16": {"drop": 0.3, "aug": True, "lr": 1e-3},
    "mobilenetv2": {"drop": 0.3, "aug": True, "lr": 1e-3},
}
dummy = tf.zeros((2, 28, 28, 1), dtype=tf.float32)

for family, cfg in shape_test_configs.items():
    tf.keras.backend.clear_session()
    gc.collect()
    model = build(family, cfg)
    salida = model(dummy, training=False)
    print(f"\n{family}: entrada {model.input_shape} -> salida {tuple(salida.shape)}",
          f"| parámetros {model.count_params():,}")
    assert model.input_shape == (None, 28, 28, 1)
    assert tuple(salida.shape) == (2, NUM_CLASSES)
    if family != "cnn":
        print(f"  backbone detectado: {backbone_de(model).name}")

print("\nPrueba superada: preprocesamiento correcto y sin incompatibilidades de forma.")


## 3. Búsqueda de hiperparámetros

Búsqueda manual estructurada con presupuesto común. Se varían capacidad, dropout, aumentación y learning rate. La selección usa mayor accuracy de validación y, como desempate, menor loss. Cada corrida inicia desde pesos nuevos. En modo rápido se emplea un subconjunto estratificado únicamente para depuración.


In [ ]:
if MODO_RAPIDO:
    idx, _ = train_test_split(np.arange(len(y_train)), train_size=12000,
                              random_state=SEED, stratify=y_train)
    ds_busqueda = make_ds(x_train[idx], y_train[idx], training=True)
    print("MODO_RAPIDO: la búsqueda usa 12 000 imágenes y", EPOCHS_SEARCH,
          "épocas. Estos números son de depuración, no sirven para el informe.")
else:
    ds_busqueda = DS["train"]

spaces = {
    "cnn": [
        {"filters": (32, 64), "drop": 0.0, "aug": False, "lr": 1e-3},
        {"filters": (32, 64), "drop": 0.3, "aug": True, "lr": 1e-3},
        {"filters": (64, 128), "drop": 0.3, "aug": True, "lr": 1e-3},
        {"filters": (32, 64), "drop": 0.3, "aug": True, "lr": 1e-4},
    ],
    "vgg16": [
        {"drop": 0.2, "aug": False, "lr": 1e-3},
        {"drop": 0.4, "aug": True, "lr": 1e-3},
        {"drop": 0.4, "aug": True, "lr": 1e-4},
    ],
    "mobilenetv2": [
        {"drop": 0.2, "aug": False, "lr": 1e-3},
        {"drop": 0.3, "aug": True, "lr": 1e-3},
        {"drop": 0.3, "aug": True, "lr": 1e-4},
    ],
}

rows, search_hist = [], {}
for f, configs in spaces.items():
    for run, cfg in enumerate(configs):
        tf.keras.backend.clear_session()
        gc.collect()
        seed_all()
        m = build(f, cfg)
        compile_m(m, cfg["lr"])
        t = time.perf_counter()
        h = m.fit(ds_busqueda, validation_data=DS["val"], epochs=EPOCHS_SEARCH,
                  callbacks=cbs(), verbose=2)
        b = int(np.argmin(h.history["val_loss"]))
        rows.append({
            "family": f,
            "run": run,
            "config": json.dumps(cfg),
            "best_epoch": b + 1,
            "val_loss": h.history["val_loss"][b],
            "val_accuracy": h.history["val_accuracy"][b],
            "train_accuracy": h.history["accuracy"][b],
            "gap": h.history["accuracy"][b] - h.history["val_accuracy"][b],
            "seconds": time.perf_counter() - t,
            "params": int(m.count_params()),
            "trainable": int(sum(np.prod(v.shape) for v in m.trainable_weights)),
        })
        search_hist[f"{f}_{run}"] = h.history
        # Se guarda en cada vuelta: si Colab se desconecta no se pierde la búsqueda.
        pd.DataFrame(rows).to_csv(RUTA_RES / "experimentos.csv", index=False)
        with open(RUTA_RES / "historias_busqueda.json", "w", encoding="utf-8") as fh:
            json.dump(search_hist, fh)

search = pd.DataFrame(rows).sort_values(["family", "val_accuracy"], ascending=[True, False])
display(search)


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, (f, g) in zip(ax, search.groupby("family")):
    g = g.sort_values("run")
    a.bar(g.run.astype(str), g.val_accuracy)
    a.set_title(f)
    a.set_xlabel("experimento")
    a.set_ylabel("accuracy validación")
plt.tight_layout()
plt.savefig(RUTA_RES / "busqueda.png", dpi=180, bbox_inches="tight")
plt.show()


## 4. Entrenamiento final y fine-tuning

In [ ]:
best = {f: json.loads(search[search.family == f]
                      .sort_values(["val_accuracy", "val_loss"], ascending=[False, True])
                      .iloc[0].config) for f in spaces}
print(json.dumps(best, indent=2))

models, hist, times = {}, {}, []
for f, cfg in best.items():
    tf.keras.backend.clear_session()
    gc.collect()
    seed_all()
    m = build(f, cfg)
    compile_m(m, cfg["lr"])
    t = time.perf_counter()
    h = m.fit(DS["train"], validation_data=DS["val"], epochs=EPOCHS_FINAL,
              callbacks=cbs("final_" + f), verbose=2)
    models[f] = m
    hist[f] = dict(h.history)
    times.append({"family": f, "stage": "base", "seconds": time.perf_counter() - t})

# Fine-tuning solo para las dos preentrenadas: se descongelan las últimas capas del backbone
# con un learning rate diez veces menor. La CNN desde cero no tiene esta fase, y eso hay que
# decirlo en el informe al comparar tiempos.
for f, n in {"vgg16": 4, "mobilenetv2": 20}.items():
    m = models[f]
    base = backbone_de(m)
    base.trainable = True
    for capa in base.layers[:-n]:
        capa.trainable = False
    for capa in base.layers:
        if isinstance(capa, tf.keras.layers.BatchNormalization):
            capa.trainable = False
    compile_m(m, best[f]["lr"] / 10)
    t = time.perf_counter()
    h = m.fit(DS["train"], validation_data=DS["val"], epochs=max(3, EPOCHS_FINAL // 2),
              callbacks=cbs("finetune_" + f), verbose=2)
    for k, v in h.history.items():
        hist[f].setdefault(k, []).extend(v)
    times.append({"family": f, "stage": f"fine_tune_{n}", "seconds": time.perf_counter() - t})

# Al repositorio solo va el modelo de la CNN desde cero (unos 150 KB). VGG16 pesa ~56 MB y
# MobileNetV2 ~9 MB: quedan en RUTA_TMP por si hay que descargarlos.
models["cnn"].save(RUTA_RES / "modelo_cnn_desde_cero.keras")
for f in ("vgg16", "mobilenetv2"):
    models[f].save(RUTA_TMP / f"modelo_{f}.keras")

pd.DataFrame(times).to_csv(RUTA_RES / "tiempos.csv", index=False)
with open(RUTA_RES / "historias_final.json", "w", encoding="utf-8") as fh:
    json.dump(hist, fh)
print("\nmodelos guardados y tiempos escritos en", RUTA_RES)


In [ ]:
fig, ax = plt.subplots(3, 2, figsize=(13, 13))
for r, f in enumerate(["cnn", "vgg16", "mobilenetv2"]):
    for col, metric in enumerate(["accuracy", "loss"]):
        ax[r, col].plot(hist[f][metric], label="train")
        ax[r, col].plot(hist[f]["val_" + metric], label="val")
        ax[r, col].set_title(f"{f}: {metric}")
        ax[r, col].set_xlabel("época")
        ax[r, col].legend()
plt.tight_layout()
plt.savefig(RUTA_RES / "curvas.png", dpi=180, bbox_inches="tight")
plt.show()


## 5. Evaluación final, usada una sola vez sobre test

In [ ]:
metrics, preds, probs = [], {}, {}
for f, m in models.items():
    pr = m.predict(DS["test"], verbose=0)
    yp = pr.argmax(1)
    preds[f], probs[f] = yp, pr
    p, r, f1, _ = precision_recall_fscore_support(y_test, yp, average="macro", zero_division=0)
    metrics.append({
        "Modelo": f,
        "Accuracy": accuracy_score(y_test, yp),
        "LogLoss": log_loss(y_test, pr, labels=list(range(NUM_CLASSES))),
        "Precision_macro": p,
        "Recall_macro": r,
        "F1_macro": f1,
        "Parametros": int(m.count_params()),
        "Entrenables": int(sum(np.prod(v.shape) for v in m.trainable_weights)),
    })

met = pd.DataFrame(metrics).sort_values("F1_macro", ascending=False)
met.to_csv(RUTA_RES / "metricas_test.csv", index=False)
display(met)

fig, ax = plt.subplots(1, 3, figsize=(19, 5))
for a, f in zip(ax, ["cnn", "vgg16", "mobilenetv2"]):
    sns.heatmap(confusion_matrix(y_test, preds[f]), annot=True, fmt="d", cmap="Blues",
                cbar=False, ax=a, xticklabels=classes, yticklabels=classes)
    a.set_title(f)
    a.set_xlabel("Predicción")
    a.set_ylabel("Real")
    a.tick_params(axis="x", rotation=90)
    a.tick_params(axis="y", rotation=0)
plt.tight_layout()
plt.savefig(RUTA_RES / "confusion.png", dpi=180, bbox_inches="tight")
plt.show()

for f in models:
    reporte = pd.DataFrame(classification_report(
        y_test, preds[f], target_names=classes, output_dict=True)).T
    reporte.to_csv(RUTA_RES / f"reporte_{f}.csv")
    print(f)
    display(reporte)


## 6. Análisis de errores e incertidumbre

In [ ]:
for f in models:
    cm = confusion_matrix(y_test, preds[f])
    np.fill_diagonal(cm, 0)
    pares = np.dstack(np.unravel_index(np.argsort(cm.ravel())[::-1][:5], cm.shape))[0]
    print()
    print(f)
    for i, j in pares:
        print(f"{classes[i]} -> {classes[j]}: {cm[i, j]}")

best_f = met.iloc[0].Modelo
err = np.where(preds[best_f] != y_test)[0]
err = err[np.argsort(probs[best_f][err, preds[best_f][err]])[::-1][:15]]
fig, ax = plt.subplots(3, 5, figsize=(13, 8))
for i, a in zip(err, ax.flat):
    a.imshow(x_test[i], cmap="gray")
    a.set_title(f"R:{classes[y_test[i]]} | P:{classes[preds[best_f][i]]}", fontsize=8)
    a.axis("off")
fig.suptitle(f"{best_f}: errores más confiados", y=1.01)
plt.tight_layout()
plt.savefig(RUTA_RES / "errores.png", dpi=180, bbox_inches="tight")
plt.show()

# Intervalo de confianza del accuracy por bootstrap sobre el conjunto de test.
rng = np.random.default_rng(SEED)
B = 300 if MODO_RAPIDO else 1000
boot = []
for f in models:
    correcto = (preds[f] == y_test).astype(float)
    muestras = [correcto[rng.integers(0, len(y_test), len(y_test))].mean() for _ in range(B)]
    boot.append({"Modelo": f, "Accuracy": correcto.mean(),
                 "IC95_inf": np.percentile(muestras, 2.5),
                 "IC95_sup": np.percentile(muestras, 97.5)})
boot = pd.DataFrame(boot)
boot.to_csv(RUTA_RES / "bootstrap.csv", index=False)
display(boot)


## 7. Cómo redactar qué funcionó y qué no

1. Comparar 32–64 con 64–128 filtros: ¿la capacidad adicional mejoró validación lo suficiente para justificar tiempo y parámetros?
2. Comparar dropout 0 y 0.3, y la brecha entrenamiento-validación: ¿redujo sobreajuste o causó subajuste?
3. Comparar learning rates 1e-3 y 1e-4: discuta rapidez, estabilidad y mejor época.
4. Comparar con y sin aumentación: una convergencia inicial más lenta puede acompañar mejor generalización.
5. Para VGG16 y MobileNetV2 compare la fase congelada con fine-tuning. Analice el cambio de dominio entre ImageNet y prendas pequeñas en gris.
6. Use la matriz y los cinco pares más confundidos. No se limite al promedio global.
7. Defina “mejor” considerando accuracy/F1, tiempo, parámetros y memoria.

### Ventajas y desventajas a contrastar con los resultados
- **CNN desde cero:** entrada nativa, control y bajo coste; no reutiliza conocimiento y depende más del diseño.
- **VGG16:** filtros generales reutilizables; muy pesada y obliga a redimensionar/replicar canales.
- **MobileNetV2:** transferencia con mejor eficiencia paramétrica; el fine-tuning es sensible al learning rate y también existe cambio de dominio.



In [ ]:
resumen = (met
           .merge(pd.DataFrame(times).groupby("family", as_index=False).seconds.sum(),
                  left_on="Modelo", right_on="family")
           .drop(columns="family")
           .merge(boot[["Modelo", "IC95_inf", "IC95_sup"]], on="Modelo"))
resumen.to_csv(RUTA_RES / "resumen_final.csv", index=False)
display(resumen)

entorno = {
    "tensorflow": tf.__version__,
    "gpu": tf.config.list_physical_devices("GPU")[0].name if HAY_GPU else None,
    "modo_rapido": MODO_RAPIDO,
    "seed": SEED,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs_search": EPOCHS_SEARCH,
    "epochs_final": EPOCHS_FINAL,
    "mejores_config": best,
    "duplicados_desarrollo": int(repetidas_dev),
    "imagenes_test_repetidas_en_desarrollo": int(fuga_test),
}
with open(RUTA_RES / "entorno.json", "w", encoding="utf-8") as fh:
    json.dump(entorno, fh, indent=2, ensure_ascii=False)

print("archivos generados en results/punto2/:")
for p in sorted(RUTA_RES.iterdir()):
    if p.is_file():
        print(f"  {p.name:<34} {p.stat().st_size / 1024:8.1f} KB")

print("\nPara dejar esto en el repo desde Colab:")
print("  1. Archivo > Guardar una copia en GitHub (rama punto2), para subir el notebook corrido.")
print("  2. Hacer commit de results/punto2/ desde una celda con git, o descargar la carpeta")
print("     y commitearla en local.")


## Referencias sugeridas en formato IEEE

[1] H. Xiao, K. Rasul, and R. Vollgraf, “Fashion-MNIST: a Novel Image Dataset for Benchmarking Machine Learning Algorithms,” arXiv:1708.07747, 2017.  
[2] K. Simonyan and A. Zisserman, “Very Deep Convolutional Networks for Large-Scale Image Recognition,” ICLR, 2015.  
[3] M. Sandler et al., “MobileNetV2: Inverted Residuals and Linear Bottlenecks,” CVPR, 2018.  
[4] TensorFlow, “Transfer learning and fine-tuning,” documentación oficial.
